In [ ]:
import pandas as pd
import numpy as np
import math
import json
from google.colab import files

In [ ]:
# Upload Kaggle CSV
# uploaded = files.upload()
# df = pd.read_csv(list(uploaded.keys())[0])

# Sample data for testing
sample = [['AIIMS Delhi',28.5672,77.2100,'Sri Aurobindo Marg New Delhi','011-26588500','New Delhi','Delhi','Government'],['Safdarjung Hospital',28.5689,77.2058,'Ring Road New Delhi','011-26165060','New Delhi','Delhi','Government'],['Apollo Hospital Delhi',28.5437,77.2780,'Sarita Vihar New Delhi','011-71791090','New Delhi','Delhi','Private'],['KEM Hospital Mumbai',18.9013,72.8345,'Acharya Donde Marg Mumbai','022-24107000','Mumbai','Maharashtra','Government'],['NIMHANS Bangalore',12.9434,77.5954,'Hosur Road Bangalore','080-46110007','Bangalore','Karnataka','Government']]
df = pd.DataFrame(sample, columns=['name','latitude','longitude','address','phone','city','state','type'])
print(df.shape)
df.head()

In [ ]:
def preprocess(df):
    df = df.copy()
    df.columns = [c.lower().strip().replace(' ','_') for c in df.columns]
    rename = {'hospital_name':'name','lat':'latitude','lon':'longitude','lng':'longitude','contact':'phone','facility_name':'name'}
    df = df.rename(columns=rename)
    for col in ['address','phone','city','state','type']:
        if col not in df.columns: df[col] = ''
    df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
    df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
    df = df.dropna(subset=['latitude','longitude'])
    df = df[(df['latitude'] != 0) & (df['longitude'] != 0)]
    df['name'] = df['name'].fillna('Unknown').astype(str).str.strip()
    return df[['name','latitude','longitude','address','phone','city','state','type']].reset_index(drop=True)

df_clean = preprocess(df)
print(f'Hospitals after cleaning: {len(df_clean)}')
df_clean.head()

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1,lon1,lat2,lon2 = map(math.radians,[lat1,lon1,lat2,lon2])
    a = math.sin((lat2-lat1)/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin((lon2-lon1)/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def find_nearest(df, user_lat, user_lon, n=5):
    df = df.copy()
    df['distance_km'] = df.apply(lambda r: haversine(user_lat, user_lon, r['latitude'], r['longitude']), axis=1)
    df['eta_minutes'] = (df['distance_km'] / 40 * 60).round(1)
    return df.sort_values('distance_km').head(n)

result = find_nearest(df_clean, 28.6139, 77.2090)
print(result[['name','distance_km','eta_minutes']])

In [ ]:
df_clean.to_csv('hospitals.csv', index=False)
print(f'Exported {len(df_clean)} hospitals')
files.download('hospitals.csv')